In [37]:
"""
- 输入：把 30x30 迷宫展平成长度 900 的一维向量（每个格子映射到 0~4）
- 网络：900 -> 4096 -> 512 -> 4（ReLU）
- 训练：用 train_data.csv + train_answer.csv 做回归（MSE），Adam，小批量（PyTorch）
- 输出：对 test_data.csv 预测，写 result.csv（每行 4 个实数）

用法：
    python baseline.py train_data.csv train_answer.csv test_data.csv result.csv
"""

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import zipfile
import os
import random

seed = 42

random.seed(seed)                  # Python built-in random
np.random.seed(seed)               # NumPy
torch.manual_seed(seed)            # PyTorch (CPU)
torch.cuda.manual_seed(seed)       # PyTorch (single GPU)
torch.cuda.manual_seed_all(seed)   # PyTorch (all GPUs)

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

N = 30
D = N * N

TRAIN_PATH = "/bohr/train-abk9/v1/"  # 训练集路径

train_x_path = TRAIN_PATH + "train_data.csv"
train_y_path = TRAIN_PATH + "train_answer.csv"

In [52]:
from collections import deque

def index_to_coordinates(index, width):
    """将一维索引转换为二维坐标"""
    return index // width, index % width

def coordinates_to_index(x, y, width):
    """将二维坐标转换为一维索引"""
    return x * width + y

def bfs_shortest_path_length(map_1d, width, start, end):
    queue = deque([start])  # 初始化队列
    visited = set()  # 记录已访问的节点
    visited.add(start)  # 将起点加入已访问集合
    distance = {start: 0}  # 记录到达每个节点的距离

    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # 上下左右移动

    while queue:
        current = queue.popleft()
        # 如果到达终点，返回路径长度（不包括起点和终点）
        if current == end:
            return distance[current] - 1  # 减去1以排除起点
        # 获取当前坐标
        x, y = index_to_coordinates(current, width)
        # 遍历四个方向
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < width and 0 <= ny < width:  # 确保在边界内
                neighbor_index = coordinates_to_index(nx, ny, width)
                if map_1d[neighbor_index] != 1 and neighbor_index not in visited:  # 不能通行的格子
                    visited.add(neighbor_index)
                    queue.append(neighbor_index)
                    distance[neighbor_index] = distance[current] + 1  # 更新距离

    return -1  # 如果没有路径，返回 -1

In [51]:
def replace_twos(sequence, ratio):
    # 将序列转换为 NumPy 数组
    arr = np.array(sequence)
    # 找到所有值为 2 的索引
    indices = np.where(arr == 2)[0]
    # 计算需要替换为 0 的数量
    num_to_replace = int(len(indices) * ratio)
    # 随机选择要替换为 0 的索引
    if num_to_replace > 0:
        replace_indices = np.random.choice(indices, size=num_to_replace, replace=False)
        arr[replace_indices] = 0  # 替换为 0
    # 将其余的 2 替换为 1
    arr[arr == 2] = 1
    return arr.tolist()

In [53]:
def findpoint(line_map, num):
    idx = line_map.index(num)
    return idx//30, idx%30

def countpoint(line_map, num):
    return line_map.count(num)

def countrowpoint(line_map, row, num):
    return line_map[row*30:(row+1)*30].count(num)

def countcolpoint(line_map, col, num):
    cnt=0
    for i in range(0,30):
        idx = i*30 + col
        if line_map[idx] == num:
            cnt += 1
    return cnt

def median_pos(line_map, num):
    pos = []
    for i in range(len(line_map)):
        if line_map[i] == num:
            pos.append(i)

    pos = np.array(pos)
    res = float(np.median(pos))
    return [res//30, res%30]

def mean_pos(line_map, num):
    pos = []
    for i in range(len(line_map)):
        if line_map[i] == num:
            pos.append(i)

    pos = np.array(pos)
    res = float(np.mean(pos))
    return [res//30, res%30]

def process(line_map):
    res = []
    sx, sy = findpoint(line_map, 3)
    tx, ty = findpoint(line_map, 4)
    res.extend([sx, sy, tx, ty])
    res.extend([countpoint(line_map, 0), countpoint(line_map, 1), countpoint(line_map, 2)])
    for i in range(0, 30, 2): #row count element
        for num in range(0,3):
            res.append(countrowpoint(line_map, i, num))

    for i in range(0, 30, 2): #col count element
        for num in range(0,3):
            res.append(countcolpoint(line_map, i, num))

    for num in range(0,3):
        res.extend(median_pos(line_map, num))
        res.extend(mean_pos(line_map, num))

    for ratio in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]:
        temp = replace_twos(line_map, ratio)
        res.append(bfs_shortest_path_length(temp, 30, temp.index(3), temp.index(4)))
    
    return res

In [68]:
def flip_horizontal(map_1d, width):
    # 将一维数组重塑为二维数组
    map_1d = np.array(map_1d)
    map_2d = map_1d.reshape((width, width))
    # 左右翻转每一行
    flipped = np.flip(map_2d, axis=1)
    return list(flipped.flatten())  # 再次摊平为一维数组

# 上下翻转
def flip_vertical(map_1d, width):
    # 将一维数组重塑为二维数组
    map_1d = np.array(map_1d)
    map_2d = map_1d.reshape((width, width))
    # 上下翻转每一列
    flipped = np.flip(map_2d, axis=0)
    return list(flipped.flatten())

In [70]:
def make_features(path: str) -> np.ndarray:
    char_id = {".": 0, "#": 1, "?": 2, "S": 3, "T": 4}

    xs = []
    with open(path, "r", encoding="utf-8-sig") as f:
        for line in f:
            if line.strip():
                transformed_line = [char_id[c] for c in line.strip()]
                res = []
                res.extend(process(transformed_line))
                res.extend(process(flip_horizontal(transformed_line, 30)))
                res.extend(process(flip_vertical(transformed_line, 30)))
                xs.append(res)

    print(xs[0])
    return np.asarray(xs, dtype=np.float32)

# make_features(train_x_path)

In [69]:
def read_y(path: str) -> np.ndarray:
    return np.loadtxt(path, delimiter=",", dtype=np.float32, encoding="utf-8-sig")

# def train_model(train_x_path: str, train_y_path: str, epochs: int) -> nn.Module:
#     # 输入缩放到约 [0,1]，标签除以 900 后回归，预测时再乘回原量纲。
#     x_train = torch.from_numpy(encode_lines(train_x_path) / 4.0)
#     y_train = torch.from_numpy(read_y(train_y_path) / 900.0)

#     torch.manual_seed(0)
#     model = nn.Sequential(
#         nn.Linear(D, 4096), nn.ReLU(),
#         nn.Linear(4096, 512), nn.ReLU(),
#         nn.Linear(512, 4),
#     )
#     loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
#     opt = torch.optim.Adam(model.parameters(), lr=1e-3)

#     model.train()
#     for ep in range(1, epochs + 1):
#         total = 0.0
#         for xb, yb in loader:
#             pred = model(xb)
#             loss = nn.functional.mse_loss(pred, yb)
#             opt.zero_grad()
#             loss.backward()
#             opt.step()
#             total += float(loss.item()) * xb.shape[0]
#         print(f"epoch {ep}/{epochs}  mse={total / len(x_train):.6f}")
#     return model

In [71]:
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

X = make_features(train_x_path)
y = read_y(train_y_path)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [78]:
model = MultiOutputRegressor(xgb.XGBRegressor())
model = model.fit(X_train, y_train)

y_pred = model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(mae, r2)

In [79]:
final_model = MultiOutputRegressor(xgb.XGBRegressor())
final_model = final_model.fit(X, y)

In [74]:
def predict(model: nn.Module, test_x_path: str) -> np.ndarray:
    x_test = make_features(test_x_path)
    return model.predict(x_test)

In [80]:
if os.environ.get("DATA_PATH"):
    DATA_PATH = os.environ.get("DATA_PATH") + "/"  # 测试集路径
else:
    DATA_PATH = "/bohr/mazeval-7zx2/v1/"  # 本地测试回退

# 测试集
testA_path = DATA_PATH + "val_data.csv"
testB_path = DATA_PATH + "test_data.csv"

#分别预测
pred_A = predict(final_model, testA_path)
pred_B = predict(final_model, testB_path)

#合并预测结果

submissionA = pd.DataFrame(pred_A)
submissionA.to_csv("./submission_val.csv", index=False, header=False)

submissionB = pd.DataFrame(pred_B)
submissionB.to_csv("./submission_test.csv", index=False, header=False)

files_to_zip = ['./submission_val.csv', './submission_test.csv']
zip_filename = 'submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')